In [ ]:
!pip install pyspark

In [ ]:
# 2. Import Required Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType
from pyspark import StorageLevel

In [ ]:
# 3. Create Spark Session
spark = SparkSession.builder \
    .appName("SparkSQL_Demo") \
    .getOrCreate()

In [ ]:
# Create Sample DataFrames
# ================================
data1 = [
    (1, "Pinky", 28, "HR"),
    (2, "Golu", 35, "IT"),
    (3, "Teju", 29, "IT"),
    (4, "Gayu", 40, "Finance"),
    (5, "Suraj", 30, "HR")
]
columns1 = ["ID", "Name", "Age", "Dept"]

In [ ]:
# createDataFrame()
df1 = spark.createDataFrame(data1, columns1)
print("Initial DataFrame:")
df1.show()

Initial DataFrame:
+---+-----+---+-------+
| ID| Name|Age|   Dept|
+---+-----+---+-------+
|  1|Pinky| 28|     HR|
|  2| Golu| 35|     IT|
|  3| Teju| 29|     IT|
|  4| Gayu| 40|Finance|
|  5|Suraj| 30|     HR|
+---+-----+---+-------+



In [ ]:
# where() & filter()
print("Employees in IT Dept (where):")
df1.where(col("Dept") == "IT").show()

Employees in IT Dept (where):
+---+----+---+----+
| ID|Name|Age|Dept|
+---+----+---+----+
|  2|Golu| 35|  IT|
|  3|Teju| 29|  IT|
+---+----+---+----+



In [ ]:
print("Employees age > 30 (filter):")
df1.filter(col("Age") > 30).show()


Employees age > 30 (filter):
+---+----+---+-------+
| ID|Name|Age|   Dept|
+---+----+---+-------+
|  2|Golu| 35|     IT|
|  4|Gayu| 40|Finance|
+---+----+---+-------+



In [ ]:
# withColumn()
df2 = df1.withColumn("Age+5", col("Age") + 5)
print("withColumn (Age+5):")
df2.show()

withColumn (Age+5):
+---+-----+---+-------+-----+
| ID| Name|Age|   Dept|Age+5|
+---+-----+---+-------+-----+
|  1|Pinky| 28|     HR|   33|
|  2| Golu| 35|     IT|   40|
|  3| Teju| 29|     IT|   34|
|  4| Gayu| 40|Finance|   45|
|  5|Suraj| 30|     HR|   35|
+---+-----+---+-------+-----+



In [ ]:
# withColumnRenamed()
df3 = df2.withColumnRenamed("Age+5", "AgeAfter5Years")
print("withColumnRenamed:")
df3.show()

withColumnRenamed:
+---+-----+---+-------+--------------+
| ID| Name|Age|   Dept|AgeAfter5Years|
+---+-----+---+-------+--------------+
|  1|Pinky| 28|     HR|            33|
|  2| Golu| 35|     IT|            40|
|  3| Teju| 29|     IT|            34|
|  4| Gayu| 40|Finance|            45|
|  5|Suraj| 30|     HR|            35|
+---+-----+---+-------+--------------+



In [ ]:
# drop()
df4 = df3.drop("AgeAfter5Years")
print("After drop column:")
df4.show()

After drop column:
+---+-----+---+-------+
| ID| Name|Age|   Dept|
+---+-----+---+-------+
|  1|Pinky| 28|     HR|
|  2| Golu| 35|     IT|
|  3| Teju| 29|     IT|
|  4| Gayu| 40|Finance|
|  5|Suraj| 30|     HR|
+---+-----+---+-------+



In [ ]:
# distinct()
print("Distinct Departments:")
df1.select("Dept").distinct().show()

Distinct Departments:
+-------+
|   Dept|
+-------+
|     HR|
|     IT|
|Finance|
+-------+



In [ ]:
# groupBy()
print("Count by Department:")
df1.groupBy("Dept").count().show()


Count by Department:
+-------+-----+
|   Dept|count|
+-------+-----+
|     HR|    2|
|     IT|    2|
|Finance|    1|
+-------+-----+



In [ ]:
# Join Example
# ================================
data2 = [
    (1, "Mumbai"),
    (2, "Delhi"),
    (3, "Bangalore"),
    (6, "Chennai")
]
columns2 = ["ID", "City"]

df_address = spark.createDataFrame(data2, columns2)
print("Join (inner join on ID):")
df1.join(df_address, on="ID", how="inner").show()

Join (inner join on ID):
+---+-----+---+----+---------+
| ID| Name|Age|Dept|     City|
+---+-----+---+----+---------+
|  1|Pinky| 28|  HR|   Mumbai|
|  2| Golu| 35|  IT|    Delhi|
|  3| Teju| 29|  IT|Bangalore|
+---+-----+---+----+---------+



In [ ]:
# map() vs mapPartitions()
# ================================
rdd = df1.rdd

# map() applies function to each row individually
mapped_rdd = rdd.map(lambda row: (row.Name, row.Age * 2))
print("map() result:", mapped_rdd.collect())

# mapPartitions() applies function to entire partition at once
def process_partition(iterator):
    result = []
    for row in iterator:
        result.append((row.Name.upper(), row.Age + 10))
    return result

mapped_partitions_rdd = rdd.mapPartitions(process_partition)
print("mapPartitions() result:", mapped_partitions_rdd.collect())

map() result: [('Pinky', 56), ('Golu', 70), ('Teju', 58), ('Gayu', 80), ('Suraj', 60)]
mapPartitions() result: [('PINKY', 38), ('GOLU', 45), ('TEJU', 39), ('GAYU', 50), ('SURAJ', 40)]


In [ ]:
# foreach() vs foreachPartition()
# ================================
print("foreach() example:")
rdd.foreach(lambda row: print(f"Name: {row.Name}"))

print("foreachPartition() example:")
def print_partition(partition):
    for row in partition:
        print(f"(Partition) Name: {row.Name}")
rdd.foreachPartition(print_partition)


foreach() example:
foreachPartition() example:


In [ ]:
# pivot()
# ================================
data_pivot = [
    ("Krish", "Math", 80),
    ("Krish", "English", 90),
    ("Bhakti", "Math", 75),
    ("Bhakti", "English", 85)
]
df_pivot = spark.createDataFrame(data_pivot, ["Name", "Subject", "Score"])
pivot_df = df_pivot.groupBy("Name").pivot("Subject").sum("Score")
print("Pivot result:")
pivot_df.show()

Pivot result:
+------+-------+----+
|  Name|English|Math|
+------+-------+----+
|Bhakti|     85|  75|
| Krish|     90|  80|
+------+-------+----+



In [ ]:
# union()
# ================================
df_extra = spark.createDataFrame([(6, "Frank", 33, "IT")], columns1)
union_df = df1.union(df_extra)
print("Union result:")
union_df.show()

Union result:
+---+-----+---+-------+
| ID| Name|Age|   Dept|
+---+-----+---+-------+
|  1|Pinky| 28|     HR|
|  2| Golu| 35|     IT|
|  3| Teju| 29|     IT|
|  4| Gayu| 40|Finance|
|  5|Suraj| 30|     HR|
|  6|Frank| 33|     IT|
+---+-----+---+-------+



In [ ]:
# collect()
# ================================
print("Collect() result:")
print(df1.collect())


Collect() result:
[Row(ID=1, Name='Pinky', Age=28, Dept='HR'), Row(ID=2, Name='Golu', Age=35, Dept='IT'), Row(ID=3, Name='Teju', Age=29, Dept='IT'), Row(ID=4, Name='Gayu', Age=40, Dept='Finance'), Row(ID=5, Name='Suraj', Age=30, Dept='HR')]


In [ ]:
# cache() & persist()
# ================================
cached_df = df1.cache()
persisted_df = df1.persist(StorageLevel.MEMORY_AND_DISK)

print("Cached and Persisted DataFrames ready.")
cached_df.show()

Cached and Persisted DataFrames ready.
+---+-----+---+-------+
| ID| Name|Age|   Dept|
+---+-----+---+-------+
|  1|Pinky| 28|     HR|
|  2| Golu| 35|     IT|
|  3| Teju| 29|     IT|
|  4| Gayu| 40|Finance|
|  5|Suraj| 30|     HR|
+---+-----+---+-------+



In [ ]:
# udf()
# ================================
def age_category(age):
    if age < 30:
        return "Young"
    elif age < 40:
        return "Middle-aged"
    else:
        return "Senior"

age_category_udf = udf(age_category, StringType())
df_with_category = df1.withColumn("Category", age_category_udf(col("Age")))
print("UDF applied result:")
df_with_category.show()

UDF applied result:
+---+-----+---+-------+-----------+
| ID| Name|Age|   Dept|   Category|
+---+-----+---+-------+-----------+
|  1|Pinky| 28|     HR|      Young|
|  2| Golu| 35|     IT|Middle-aged|
|  3| Teju| 29|     IT|      Young|
|  4| Gayu| 40|Finance|     Senior|
|  5|Suraj| 30|     HR|Middle-aged|
+---+-----+---+-------+-----------+



In [ ]:
# Stop Spark Session
# ================================
spark.stop()